# Segmentation cardiaque (ACDC) : baseline UNet

**Objectif du notebook :** entraîner un UNet de base qui segmente 3 structures du cœur (classes 1, 2, 3, plus le fond = classe 0) sur des coupes IRM, choisir le meilleur learning rate, puis évaluer le modèle retenu.

**Plan :**
1. Préparation : imports, graine aléatoire, fonction d'entraînement `runTraining`.
2. Balayage du learning rate : pour chacun des 6 lr, on entraîne le UNet (100 époques), puis on évalue **une seule fois** son meilleur modèle (toutes les métriques + ses figures).
3. Synthèse : tableau comparatif lr × modèles, modèle retenu, figure de sensibilité au lr. Cette partie ne fait que lire les fichiers produits en 2.

---
## 1. Préparation

**Imports.** On charge PyTorch, le chargeur de données du prof (`medicalDataLoader`), la barre de progression, nos utilitaires (`utils_Nadir` : Dice, conversion des masques…) et notre UNet (`UNet_Base_Nadir`).

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms
from progressBar import printProgressBar

import medicalDataLoader
import argparse
from utils_Nadir import *

from UNet_Base_Nadir import *
import random
import torch
import pdb

**Reproductibilité.** `set_seed` fixe toutes les sources de hasard (Python, NumPy, PyTorch CPU et GPU, cuDNN). Avec la même graine, deux entraînements partent des mêmes poids initiaux et voient les batchs dans le même ordre : on peut donc comparer les learning rates de façon équitable (seul le lr change).

In [ ]:
def set_seed(seed=0):
    random.seed(seed)                 # module random de Python
    np.random.seed(seed)              # NumPy
    torch.manual_seed(seed)           # PyTorch : poids initiaux, mélange des batchs, dropout
    torch.cuda.manual_seed_all(seed)  # idem sur GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

**Masquer les avertissements.** Purement cosmétique : évite que les messages de dépréciation de PyTorch (`upsample`, `kaiming_normal`…) noient les sorties de l'entraînement. Ne change aucun résultat.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

**Fonction d'entraînement `runTraining(lr, epoch, seed, modelName)`.** Cette cellule ne fait que *définir* la fonction : rien ne s'entraîne ici. À chaque appel, elle :

1. charge les données d'entraînement (mélangées) et de validation (ordre fixe), sous forme de tenseurs, sans augmentation ;
2. crée un UNet à 4 sorties (fond + 3 classes) et l'optimiseur Adam avec le `lr` demandé ;
3. pour chaque époque :
   - **entraînement** : prédiction, perte d'entropie croisée (CE) entre la prédiction et le masque converti en classes 0..3 (`getTargetSegmentation`), rétropropagation, mise à jour des poids ;
   - **validation** : CE de validation et Dice par classe, calculé sur toute la validation d'un coup ;
   - si le Dice moyen de validation est le meilleur vu jusqu'ici, les poids sont sauvegardés dans `models/<modelName>/best_model`. On garde donc la **meilleure époque**, pas la dernière ;
4. renvoie un dictionnaire `history` (pertes, CE et Dice par époque, meilleur Dice, meilleure époque) qui sert ensuite à tous les tableaux et figures.

In [ ]:
def runTraining(lr=0.01, epoch=50, seed=0, modelName='Test_Model'):
    set_seed(seed)
    print('-' * 40)
    print('~~~~~~~~  Starting the training... ~~~~~~')
    print('-' * 40)

    ## DEFINE HYPERPARAMETERS (batch_size > 1)
    batch_size = 8
    batch_size_val = 8
    
    root_dir = './Data/'

    print(' Dataset: {} '.format(root_dir))

    ## DEFINE THE TRANSFORMATIONS TO DO AND THE VARIABLES FOR TRAINING AND VALIDATION
    
    transform = transforms.Compose([
        transforms.ToTensor()
    ])

    mask_transform = transforms.Compose([
        transforms.ToTensor()
    ])

    train_set_full = medicalDataLoader.MedicalImageDataset('train',
                                                      root_dir,
                                                      transform=transform,
                                                      mask_transform=mask_transform,
                                                      augment=False,
                                                      equalize=False)

    train_loader_full = DataLoader(train_set_full,
                              batch_size=batch_size,
                              worker_init_fn=np.random.seed(0),
                              num_workers=0,
                              shuffle=True)


    val_set = medicalDataLoader.MedicalImageDataset('val',
                                                    root_dir,
                                                    transform=transform,
                                                    mask_transform=mask_transform,
                                                    equalize=False)

    val_loader = DataLoader(val_set,
                            batch_size=batch_size_val,
                            worker_init_fn=np.random.seed(0),
                            num_workers=0,
                            shuffle=False)


    ## INITIALIZE YOUR MODEL
    num_classes = 4 # NUMBER OF CLASSES

    print("~~~~~~~~~~~ Creating the UNet model ~~~~~~~~~~")
    print(" Model Name: {}".format(modelName))

    ## CREATION OF YOUR MODEL
    net = UNet(num_classes)

    print("Total params: {0:,}".format(sum(p.numel() for p in net.parameters() if p.requires_grad)))

    # DEFINE YOUR OUTPUT COMPONENTS (e.g., SOFTMAX, LOSS FUNCTION, ETC)
    softMax = nn.Softmax(dim=1)
    CE_loss = nn.CrossEntropyLoss()

    ## PUT EVERYTHING IN GPU RESOURCES    
    if torch.cuda.is_available():
        net.cuda()
        softMax.cuda()
        CE_loss.cuda()

    ## DEFINE YOUR OPTIMIZER
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)

    ### To save statistics ####
    lossTotalTraining = []
    Best_loss_val = 1000
    BestEpoch = 0
    Best_dice_val = 0
    history = {'lr': lr, 'seed': seed, 'train_loss': [], 'val_ce': [], 'val_dice': [], 'val_dice_mean': []}
    
    directory = 'Results/Statistics/' + modelName

    print("~~~~~~~~~~~ Starting the training ~~~~~~~~~~")
    if os.path.exists(directory)==False:
        os.makedirs(directory)

    ## START THE TRAINING
    
    ## FOR EACH EPOCH
    for i in range(epoch):
        net.train()
        lossEpoch = []
        DSCEpoch = []
        DSCEpoch_w = []
        num_batches = len(train_loader_full)
        
        ## FOR EACH BATCH
        for j, data in enumerate(train_loader_full):
            ### Set to zero all the gradients
            net.zero_grad()
            optimizer.zero_grad()

            ## GET IMAGES, LABELS and IMG NAMES
            images, labels, img_names = data

            ### From numpy to torch variables
            labels = to_var(labels)
            images = to_var(images)

            ################### Train ###################
            #-- The CNN makes its predictions (forward pass)
            net_predictions = net(images)

            #-- Compute the losses --#
            # THIS FUNCTION IS TO CONVERT LABELS TO A FORMAT TO BE USED IN THIS CODE
            segmentation_classes = getTargetSegmentation(labels)
            
            # COMPUTE THE LOSS
            CE_loss_value = CE_loss(net_predictions, segmentation_classes) # XXXXXX and YYYYYYY are your inputs for the CE
            lossTotal = CE_loss_value

            # DO THE STEPS FOR BACKPROP (two things to be done in pytorch)
            lossTotal.backward()
            optimizer.step()
            
            # THIS IS JUST TO VISUALIZE THE TRAINING 
            lossEpoch.append(lossTotal.cpu().data.numpy())
            printProgressBar(j + 1, num_batches,
                             prefix="[Training] Epoch: {} ".format(i),
                             length=15,
                             suffix=" Loss: {:.4f}, ".format(lossTotal))

        lossEpoch = np.asarray(lossEpoch)
        lossEpoch = lossEpoch.mean()

        lossTotalTraining.append(lossEpoch)

        printProgressBar(num_batches, num_batches,
                             done="[Training] Epoch: {}, LossG: {:.4f}".format(i,lossEpoch))
        
        # ---- VALIDATION ----
        net.eval()
        all_preds = []
        all_gt = []
        ce_val = []
        with torch.no_grad():
            for data in val_loader:
                images, labels, img_names = data
                images = to_var(images)
                labels = to_var(labels)
                net_predictions = net(images)
                gt = getTargetSegmentation(labels)
                ce_val.append(CE_loss(net_predictions, gt).item())
                all_preds.append(torch.argmax(net_predictions, dim=1))
                all_gt.append(gt)
        dices_val = dice_per_class(torch.cat(all_preds), torch.cat(all_gt))
        print("[Validation] CE : {:.4f}  Dice classes 1/2/3 : {}  moyenne : {:.4f}".format(
            np.mean(ce_val), np.round(dices_val, 4), np.mean(dices_val)))

        history['train_loss'].append(float(lossEpoch))
        history['val_ce'].append(float(np.mean(ce_val)))
        history['val_dice'].append(dices_val)
        history['val_dice_mean'].append(float(np.mean(dices_val)))
        
        ## THIS IS HOW YOU WILL SAVE THE TRAINED MODELS AFTER EACH EPOCH. 
        ## WARNING!!!!! YOU DON'T WANT TO SAVE IT AT EACH EPOCH, BUT ONLY WHEN THE MODEL WORKS BEST ON THE VALIDATION SET!!
        if np.mean(dices_val) > Best_dice_val:
            Best_dice_val = np.mean(dices_val)
            BestEpoch = i
            if not os.path.exists('./models/' + modelName):
                os.makedirs('./models/' + modelName)
            torch.save(net.state_dict(), './models/' + modelName + '/best_model')
            print("  -> Nouveau meilleur modèle : Dice {:.4f} à l'époque {}".format(Best_dice_val, i))
            
        np.save(os.path.join(directory, 'Losses_Nadir.npy'), lossTotalTraining)

    history['best_dice'] = float(Best_dice_val)
    history['best_epoch'] = BestEpoch
    return history

---
## 2. Balayage du learning rate : entraînement + évaluation en une seule boucle

**Idée :** le learning rate est l'hyperparamètre qui influence le plus l'entraînement. On entraîne donc le même UNet, avec la même graine, pour 6 valeurs réparties sur une échelle logarithmique (de 1e-4 à 3e-2), 100 époques chacune.

**Tout est calculé une seule fois, juste après chaque entraînement** (`ev.evaluer_entrainement`) :
1. `runTraining` entraîne le modèle et sauvegarde les poids de la meilleure époque dans `models/lr_<valeur>/best_model` ; l'historique va dans `Results/Statistics/lr_<valeur>/history.json` ;
2. on recharge ce `best_model`, on prédit la validation **une seule fois**, et à partir de ces prédictions on calcule toutes les métriques du tableau : CE moyenne, Dice (par classe et moyen), Hausdorff, ASD, nombre de paramètres. Elles sont écrites dans `Results/Statistics/lr_<valeur>/metrics.json` ;
3. avec ces mêmes prédictions, on trace les figures de ce modèle (courbes d'apprentissage et figure qualitative) dans `Results/Figures/lr_<valeur>/`.

Les images de validation sont chargées en mémoire une seule fois avant la boucle.

**`ENTRAINER`** : `True` lance les 6 entraînements (⚠️ long, et **écrase** les résultats actuels). `False` ne réentraîne rien : on réutilise les poids et `history.json` déjà présents et on refait seulement l'étape d'évaluation (environ une minute).

À la fin de chaque lr, une ligne résume ses métriques.

In [ ]:
import json
import importlib
import evaluation_Nadir as ev
importlib.reload(ev)   # prend en compte les modifications du module sans redémarrer le noyau

N_EPOCHS = 100                                   # balayage final de la baseline
LRS = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2]
ENTRAINER = True    # False : réutilise models/ et history.json existants (pas de réentraînement)

images_val, gt_val, noms_val = ev.charger_validation('./Data/')   # chargée une seule fois

historiques, metriques = {}, {}
for lr in LRS:
    name = ev.nom_modele(lr)                     # 'lr_<valeur>'
    if ENTRAINER:
        historiques[lr] = runTraining(lr=lr, epoch=N_EPOCHS, seed=0, modelName=name)
        with open('Results/Statistics/{}/history.json'.format(name), 'w') as f:
            json.dump(historiques[lr], f)
    else:
        historiques[lr] = ev.lire_historiques([lr])[lr]

    # Évaluation du best_model : métriques (metrics.json) + figures de ce lr
    m = ev.evaluer_entrainement(name, historiques[lr], images_val, gt_val, noms_val, seed=0)
    metriques[lr] = m
    print('>>> lr={} : meilleure époque {} | CE {:.4f} | Dice {:.4f} | HD {:.2f} px | ASD {:.2f} px'.format(
        lr, m['meilleure_epoque'], m['ce_moyenne'], m['dice_moyen'], m['hd_moyenne_px'], m['asd_moyenne_px']))

---
## 3. Synthèse des résultats (aucun calcul lourd)

Cette partie **ne recalcule rien** : elle lit les fichiers écrits par la boucle de la partie 2 (`history.json`, `metrics.json`, figures). Elle peut donc être relancée seule, même après un redémarrage du noyau, tant que la partie 2 a tourné une fois.

Les classes sont notées « classe 1/2/3 » (correspondance VD / myocarde / VG à vérifier visuellement).

**A. Tableau comparatif : tous les learning rates × tous les modèles.** Une ligne par couple (lr, modèle), groupée par lr, avec les métriques de validation du `best_model` :
- **CE moyenne** : entropie croisée sur tous les pixels (plus bas = mieux) ;
- **Dice moyen** : recouvrement prédiction / vérité, moyenne des 3 classes (1 = parfait, plus haut = mieux) ;
- **Hausdorff moyen** (px) : la *pire* erreur de contour, c'est-à-dire la plus grande distance entre un point d'un contour et l'autre contour (plus bas = mieux) ;
- **ASD moyenne** (px) : la distance *moyenne* entre les contours prédit et réel, symétrique (`medpy assd`). Elle est moins sensible qu'Hausdorff à un seul point aberrant (plus bas = mieux) ;
- **Nb paramètres** : complexité du modèle.

Pour l'instant seule la baseline y figure. **Pour ajouter un modèle :** l'entraîner et l'évaluer avec la même boucle en utilisant un `modelName` du type `'v1_lr_{}'.format(lr)` (et `construire=lambda: UNetV1(4)` dans `evaluer_entrainement`), puis décommenter ou ajouter sa ligne dans `MODELES`. Tant qu'il n'est pas évalué, sa ligne s'affiche avec `n/d`. Sortie : `Results/table_comparaison.csv`.

In [ ]:
# ---- A. Tableau comparatif (lr x modèles) ----
import os
import json
import importlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, Image as ImageIPy, display

import evaluation_Nadir as ev
importlib.reload(ev)
from UNet_Base_Nadir import UNet

LRS = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2]       # mêmes lr que la boucle de la partie 2
DOSSIER_RESULTATS = 'Results'

# Modèles à comparer. 'dossier' = modelName utilisé dans la boucle de la partie 2.
MODELES = [
    {'nom': 'Baseline', 'construire': lambda: UNet(4), 'dossier': 'lr_{lr}'},
    # {'nom': 'Modèle V1', 'construire': lambda: UNetV1(4), 'dossier': 'v1_lr_{lr}'},
    # {'nom': 'Modèle V2', 'construire': lambda: UNetV2(4), 'dossier': 'v2_lr_{lr}'},
    # {'nom': 'Modèle V3', 'construire': lambda: UNetV3(4), 'dossier': 'v3_lr_{lr}'},
]

historiques = ev.lire_historiques(LRS)
lignes_comp = ev.tableau_comparatif(LRS, MODELES)
ev.ecrire_csv(lignes_comp, ev.COLONNES_COMPARAISON,
              os.path.join(DOSSIER_RESULTATS, 'table_comparaison.csv'))
display(Markdown(ev.tableau_markdown(lignes_comp, ev.COLONNES_COMPARAISON,
                                     entetes=ev.ENTETES_COMPARAISON)))

**B. Modèle retenu (baseline).** Le lr retenu est celui qui a le plus grand Dice moyen de validation. On affiche son tableau détaillé **par classe** (déjà calculé dans `metrics.json`) :
- **Dice**, **Hausdorff** (moyenne et écart-type) et **ASD** pour chaque classe ;
- le nombre de coupes où le modèle « oublie » une structure présente (prédiction vide alors que la vérité ne l'est pas). Les distances ne peuvent pas être calculées sur ces coupes, d'où ce décompte à part.

Le contrôle de cohérence vérifie que le Dice recalculé à partir du `best_model` est bien égal à celui enregistré pendant l'entraînement. Ses figures sont copiées à la racine de `Results/` pour le rapport. Sorties : `Results/table_final.csv`, `Results/final_model_info.json`, `Results/fig_learning_curves.png`, `Results/fig_qualitative.png`.

In [ ]:
# ---- B. Modèle retenu : tableau détaillé par classe ----
metriques_base = {lr: ev.lire_metriques(ev.nom_modele(lr)) for lr in LRS}
meilleur_lr = max(LRS, key=lambda lr: metriques_base[lr]['dice_moyen'])
nom_retenu = ev.nom_modele(meilleur_lr)
m = metriques_base[meilleur_lr]

ev.ecrire_csv(m['par_classe'], ev.COLONNES_FINAL, os.path.join(DOSSIER_RESULTATS, 'table_final.csv'))
print('Modèle retenu : models/{}/best_model  (lr = {}, meilleure époque {} / {})'.format(
    nom_retenu, meilleur_lr, m['meilleure_epoque'], m['n_epoques']))
print('Nombre de paramètres : {:,}'.format(m['nb_parametres']))
display(Markdown(ev.tableau_markdown(m['par_classe'], ev.COLONNES_FINAL, entetes=ev.ENTETES_FINAL)))

# Contrôle de cohérence : le Dice recalculé doit égaler le meilleur Dice de history.json
print('Dice moyen recalculé : {:.6f} | history.json : {:.6f}'.format(m['dice_moyen'], m['best_dice_history']))

with open(os.path.join(DOSSIER_RESULTATS, 'final_model_info.json'), 'w', encoding='utf-8') as f:
    json.dump({'lr': meilleur_lr, 'modele': 'models/{}/best_model'.format(nom_retenu),
               'nb_parametres': m['nb_parametres'], 'meilleure_epoque': m['meilleure_epoque'],
               'n_epoques_historique': m['n_epoques']}, f, indent=2, ensure_ascii=False)
print('Figures copiées :', ev.copier_figures(nom_retenu, DOSSIER_RESULTATS))

**C. Courbes d'apprentissage du modèle retenu** (tracées pendant la partie 2, simplement affichées ici). (a) La perte d'entraînement et la CE de validation par époque : si la validation remonte pendant que l'entraînement continue de baisser, c'est du **sur-apprentissage**. (b) Le Dice de validation par classe et en moyenne. La ligne pointillée marque la meilleure époque, celle du modèle sauvegardé. Les courbes des autres lr sont dans `Results/Figures/lr_<valeur>/`.

In [ ]:
# ---- C. Courbes d'apprentissage du modèle retenu ----
display(ImageIPy(filename=os.path.join(DOSSIER_RESULTATS, 'fig_learning_curves.png'), width=900))

**D. Figure qualitative du modèle retenu** (tracée pendant la partie 2). Pour *voir* ce que fait le modèle, on a choisi 4 coupes de validation selon leur Dice : la **meilleure**, la **médiane**, la **pire** et une **au hasard** (tirage reproductible). Chaque ligne montre l'image IRM, le vrai masque, la prédiction et la carte des erreurs (pixels mal classés en magenta). Les coupes sans aucune structure sont exclues du classement.

In [ ]:
# ---- D. Figure qualitative du modèle retenu ----
display(ImageIPy(filename=os.path.join(DOSSIER_RESULTATS, 'fig_qualitative.png'), width=900))

**E. Sensibilité au learning rate.** C'est la seule figure qui compare les lr entre eux : elle ne peut donc être tracée qu'après le balayage. Elle se construit uniquement à partir des `history.json` (aucune prédiction). (a) Le meilleur Dice (par classe et en moyenne) en fonction du lr, en échelle logarithmique : cela montre quelle plage de lr fonctionne. (b) L'évolution du Dice moyen de validation au fil des époques pour chaque lr : cela montre la vitesse de convergence et la stabilité. Sortie : `Results/fig_lr_sweep.png`.

In [ ]:
# ---- E. Sensibilité au learning rate ----
fig, fichiers = ev.figure_balayage_lr(historiques, os.path.join(DOSSIER_RESULTATS, 'fig_lr_sweep.png'))
print('Écrit :', fichiers)
plt.show()

**Vérification.** Contrôle final : tous les tableaux et figures de la synthèse existent bien dans `Results/`. La cellule s'arrête avec une erreur s'il en manque un.

In [ ]:
# ---- Vérification : tous les fichiers attendus existent ----
attendus = ['table_comparaison.csv', 'table_final.csv', 'final_model_info.json',
            'fig_learning_curves.png', 'fig_qualitative.png', 'fig_lr_sweep.png']
manquants = [f for f in attendus if not os.path.exists(os.path.join(DOSSIER_RESULTATS, f))]
for f in attendus:
    print('{:28s} {}'.format(f, 'OK' if f not in manquants else 'MANQUANT'))
assert not manquants, 'Fichiers manquants : {}'.format(manquants)